In [4]:
import pandas as pd
import re
import json
import csv
import os
import glob

## Sofifa data

In [3]:
def clean_player_data(input_csv,output_csv):
    df = pd.read_csv(input_csv)

    # Helper function to safely extract regex groups
    def extract_field(pattern, text):
        match = re.search(pattern, str(text))
        return match.group(1).strip() if match else None

    # Extracting core Stats
    df['Age'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(\d+)y\.o\.', x))
    df['DOB'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'\(([A-Z][a-z]{2}\s\d{1,2},\s\d{4})\)', x))
    df['Height_cm'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(\d+)cm', x))
    df['Weight_kg'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(\d+)kg', x))
    df['Overall_Rating'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(\d+)\s*\|\s*Overall rating', x))
    df['Potential_Rating'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(\d+)(?:\s*[+-]\d+)?\s*\|\s*Potential', x))
    df['Value_EUR'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(€[\d\.KMB]+)\s*\|\s*Value', x))
    df['Wage_EUR'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'(€[\d\.KMB]+)\s*\|\s*Wage', x))
    df['Preferred_Foot'] = df['Raw_Card_Text'].apply(lambda x: extract_field(r'Preferred foot\s([A-Za-z]+)', x))

   #ATTRIBUTE EXTRACTION
    #| number Attribute Name |
    attr_pattern = r'\|\s*(\d{1,3})\s+([A-Za-z\s]+?)\s*\|'
    
    all_attributes_list = []
    
    for text in df['Raw_Card_Text']:
        matches = re.findall(attr_pattern, str(text))

        # Building a dictionary for the row, cleaning up spaces in column names
        row_attrs = {attr_name.strip().replace(' ', '_'): int(val) for val, attr_name in matches}
        all_attributes_list.append(row_attrs)
        
    attrs_df = pd.DataFrame(all_attributes_list)     # Converting the list of dictionaries into a new DataFrame

    # Combination and Clean Up
    # Concatenate the dynamic attribute columns to the main DataFrame
    df = pd.concat([df, attrs_df], axis=1)
    
    # Converting numeric columns to actual numeric types for the database
    numeric_cols = ['Age', 'Height_cm', 'Weight_kg', 'Overall_Rating', 'Potential_Rating']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Dropping raw string column
    df_cleaned = df.drop(columns=['Raw_Card_Text'])
    df_cleaned.to_csv(output_csv,index=False)
    display(df_cleaned)
    return df_cleaned



In [4]:

cleaned_df=clean_player_data("data/sofifa/sofifa_raw_data_COMPLETE.csv","data/sofifa/sofifa_cleaned.csv")

,URL,Name,Age,DOB,Height_cm,Weight_kg,Overall_Rating,Potential_Rating,Value_EUR,Wage_EUR,...,Sliding_tackle,GK_Diving,GK_Handling,GK_Kicking,GK_Positioning,GK_Reflexes,Marking,Positioning,Tactical_awareness,Tackling
0,https://sofifa.com/player/272465/caleb-roberts...,Caleb Roberts,19,"Oct 24, 2005",178,74,56,70,€350K,€600,...,42.0,14.0,6.0,13.0,13.0,7.0,NaN,NaN,NaN,NaN
1,https://sofifa.com/player/random?1775801033,Javier Villar del Fraile,22,"Mar 15, 2003",187,69,65,74,€1.6M,€2K,...,67.0,8.0,6.0,8.0,7.0,15.0,NaN,NaN,NaN,NaN
2,https://sofifa.com/player/274482/adrian-ascues...,Adrián Ascues,21,"Nov 15, 2002",181,81,65,73,€1.6M,€12K,...,41.0,8.0,14.0,6.0,11.0,8.0,NaN,NaN,NaN,NaN
3,https://sofifa.com/player/272535/josh-bailey/2...,Josh Bailey,18,"Jun 24, 2005",179,72,52,63,€160K,€500,...,44.0,9.0,11.0,11.0,8.0,10.0,NaN,NaN,NaN,NaN
4,https://sofifa.com/player/271880/nestor-jimene...,Nestor Jiménez,22,"Apr 8, 2003",175,66,61,70,€750K,€4K,...,18.0,10.0,13.0,12.0,6.0,8.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6994,https://sofifa.com/player/272834/joao-pedro-go...,João Pedro Gonçalves Neves,20,"Sep 27, 2004",174,66,87,92,€117.5M,€110K,...,82.0,8.0,7.0,5.0,13.0,13.0,NaN,NaN,NaN,NaN
6995,https://sofifa.com/player/276694/johan-manzamb...,Johan Manzambi,19,"Oct 14, 2005",183,75,75,85,€12M,€18K,...,67.0,8.0,10.0,8.0,13.0,9.0,NaN,NaN,NaN,NaN
6996,https://sofifa.com/player/263765/tom-bischof/2...,Tom Bischof,20,"Jun 28, 2005",180,75,78,86,€30.5M,€41K,...,66.0,9.0,13.0,12.0,13.0,6.0,NaN,NaN,NaN,NaN
6997,https://sofifa.com/player/234826/antonio-marti...,Antonio Martínez López,28,"Jun 30, 1997",187,83,1,76,€7.5M,€31K,...,23.0,11.0,11.0,12.0,7.0,9.0,NaN,NaN,NaN,NaN


## soccersolver data

In [2]:
json_folder = 'data/soccersolver-player-data/players' 

# The name of the final file you want to create
output_csv = 'data/soccersolver-player-data/unified/merged_players.csv'

headers = [
    "player_id", "name", "team", "team_id", "position", 
    "main_position", "age", "birth_date", "nationality", 
    "height", "preferred_foot", "shirt_number", "market_value", 
    "img_url", "profile_url", "season"
]

all_players_data = []

for filename in os.listdir(json_folder):
    if filename.endswith('.json'):
        print(f"Processing: {filename}")
        file_path = os.path.join(json_folder, filename)
        
        with open(file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)
            
            if isinstance(data, dict):
                data = [data]            # This handles both cases: if the JSON is a single player dict or a list of multiple player dicts.
                
            for player in data:
                clean_player = {key: player.get(key, '') for key in headers}        # Create a clean dictionary for each player keeping only our defined headers If a field is missing in the JSON, it defaults to an empty string ('')
                all_players_data.append(clean_player)

with open(output_csv, 'w', newline='', encoding='utf-8') as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=headers)
    writer.writeheader()
    writer.writerows(all_players_data)

print(f"\Compiled {len(all_players_data)} player records into '{output_csv}'.")

<>:35: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
<>:35: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
C:\Users\ashwy\AppData\Local\Temp\ipykernel_13932\2360835386.py:35: SyntaxWarning: "\C" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\C"? A raw string is also an option.
  print(f"\Compiled {len(all_players_data)} player records into '{output_csv}'.")


Processing: players_all_2019-2020.json
Processing: players_all_2020-2021.json
Processing: players_all_2021-2022.json
Processing: players_all_2022-2023.json
Processing: players_all_2023-2024.json
Processing: players_all_2024-2025.json
Processing: players_all_2025-2026.json
\Compiled 405290 player records into 'data/soccersolver-player-data/unified/merged_players.csv'.


In [7]:
season_files_path = 'data/soccersolver-player-data/leagues/season_files/*.json'     # Folder containing your 7 season .json files
master_file_path = 'data/soccersolver-player-data/leagues/discovered_leagues.json'    # master file with general information

output_csv = 'data/soccersolver-player-data/unified/unified_leagues.csv'  # final output file

with open(master_file_path, 'r', encoding='utf-8') as f:
    master_data = json.load(f)

if isinstance(master_data, dict):
    # Scenario A: It's a dict of dicts (e.g., {"19LA": {...}, "IT1": {...}})
    if all(isinstance(v, dict) for v in master_data.values()):
        records = list(master_data.values())
    # Scenario B: It's wrapped inside a key (e.g., {"data": [{...}, {...}]})
    else:
        records = []
        for key, value in master_data.items():
            if isinstance(value, list):
                records = value
                break
elif isinstance(master_data, list):
    # Scenario C: It's already a clean list
    records = master_data
else:
    records = [master_data]

master_df = pd.DataFrame(records)

if 'league_id' not in master_df.columns:
    print("'league_id' not found in master data.")
    print("Columns extracted:", list(master_df.columns))
    print("Sample record:", records[0] if records else "Empty")
    raise KeyError("The master JSON structure is completely unrecognized. Check the print outputs above.")

# Isolate the specific columns we want to inject
desired_columns = ['league_id', 'url_path', 'value_m', 'region', 'is_youth']
existing_columns = [col for col in desired_columns if col in master_df.columns]

master_static_data = master_df[existing_columns]

# Ingesting and Compiling Season Fact Tables 
season_files = glob.glob(season_files_path)

# List comprehension to read all JSONs into DataFrames, then concatenate them into one huge table
season_dfs = [pd.read_json(file) for file in season_files]
all_seasons_df = pd.concat(season_dfs, ignore_index=True)

# Hash Join
# merges based on the matching 'league_id'.
final_df = pd.merge(all_seasons_df, master_static_data, on='league_id', how='left')
print(final_df.head())

  league_id                 name   country     season   tier  \
0      IT3A   Serie C - Girone A     Italy  2019-2020      3   
1      CLPD      Liga de Primera     Chile  2019-2020      1   
2      PT23   Liga Revelação U23  Portugal  2019-2020  youth   
3        L3              3. Liga   Germany  2019-2020      3   
4      MLS1  Major League Soccer       USA  2019-2020      1   

   total_market_value  num_teams  num_players  average_age  \
0           106600000         20          564         24.5   
1           158330000         16          425         27.0   
2            11475000         17          527         20.2   
3           155225000         20          589         25.4   
4          1319650000         30          845         25.4   

   average_market_value most_valuable_player  \
0                189000                        
1                373000                        
2                 22000                        
3                264000                        
4 

In [8]:
final_df.to_csv(output_csv, index=False, encoding='utf-8')

print(f"Success! Processed {len(all_seasons_df)} season records and saved to {output_csv}.")

Success! Processed 525 season records and saved to data/soccersolver-player-data/unified/unified_leagues.csv.
